# Materials 02 — The energy–volume curve and the bulk modulus

The original Tutorial 2 teaches LAMMPS automation: `index` variables,
`next`/`jump` loops, and structured output files, all to scan the lattice
constant and fit an equation of state. In a notebook the loop is **just
Python** — LAMMPS runs inside it, and the data lands directly in numpy.

We scan the lattice constant of the LJ fcc crystal, record cohesive energy
vs. volume, and fit the **third-order Birch–Murnaghan equation of state** to
extract the equilibrium volume, lattice constant and bulk modulus.

In [ ]:
%pip install lammps-js matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps

lmp = await lammps(output=None)

def cell(a):
    """Build a static fcc crystal at lattice constant `a`, return (v/atom, E/atom)."""
    lmp.commands_string(f"""
clear
units lj
atom_style atomic
lattice fcc {4 / a**3:.8f}
region box block 0 3 0 3 0 3
create_box 1 box
create_atoms 1 box
mass 1 1.0
pair_style lj/cut 2.5
pair_coeff 1 1 1.0 1.0 2.5
run 0 post no
""")
    n = lmp.get_natoms()
    return lmp.get_thermo("vol") / n, lmp.get_thermo("pe")

lats = np.linspace(1.46, 1.70, 15)
data = np.array([cell(a) for a in lats])
vol, ene = data[:, 0], data[:, 1]
print("scanned", len(lats), "lattice constants")

## Fit the Birch–Murnaghan equation of state

The third-order Birch–Murnaghan energy
$E(V) = E_0 + \tfrac{9V_0B_0}{16}\{[(V_0/V)^{2/3}-1]^3 B_0' +
[(V_0/V)^{2/3}-1]^2[6-4(V_0/V)^{2/3}]\}$
looks fearsome, but substitute $x = V^{-2/3}$ and it is **exactly a cubic
polynomial in x** — so `np.polyfit` fits it without any nonlinear
optimization:

In [ ]:
x = vol ** (-2.0 / 3.0)
coeffs = np.polyfit(x, ene, 3)
poly, dpoly = np.poly1d(coeffs), np.poly1d(coeffs).deriv()

# Equilibrium: dE/dV = 0  ->  dE/dx = 0 at the physical root
roots = dpoly.r
x0 = float(min((r.real for r in roots if abs(r.imag) < 1e-9 and r.real > 0),
               key=lambda r: abs(r - x.mean())))
v0 = x0 ** (-1.5)
e0 = float(poly(x0))
a0 = (4 * v0) ** (1 / 3)             # fcc: 4 atoms per cubic cell

# Bulk modulus B0 = V d2E/dV2 at V0 (chain rule through x = V^(-2/3))
d2poly = dpoly.deriv()
dE_dx2 = float(d2poly(x0))
dx_dV = -2.0 / 3.0 * v0 ** (-5.0 / 3.0)
d2x_dV2 = 10.0 / 9.0 * v0 ** (-8.0 / 3.0)
B0 = v0 * (dE_dx2 * dx_dV**2 + float(dpoly(x0)) * d2x_dV2)

print(f"V0 = {v0:.4f} sigma^3/atom   a0 = {a0:.5f} sigma")
print(f"E0 = {e0:.5f} eps/atom       B0 = {B0:.3f} eps/sigma^3")

In [ ]:
vfine = np.linspace(vol.min(), vol.max(), 300)
plt.figure(figsize=(5.5, 3.6))
plt.plot(vol, ene, "o", label="LAMMPS")
plt.plot(vfine, poly(vfine ** (-2 / 3)), "-", label="Birch–Murnaghan fit")
plt.axvline(v0, ls="--", lw=1, color="gray")
plt.xlabel("volume per atom (σ³)"); plt.ylabel("cohesive energy (ε/atom)")
plt.legend(); plt.tight_layout(); plt.show()

lmp.close()

The fitted **a₀ matches the box-relaxation value from
[01](01-perfect-crystal.ipynb)** — a reassuring consistency check between two
independent methods, exactly as in the original tutorial. The bulk modulus
comes free with it.

**Exercises**
- Narrow the scan range around the minimum. Does B₀ change? (The fit is
  exact for BM-shaped data, but the LJ E(V) is only *approximately* BM.)
- Repeat with cutoff 3.0 σ. How much do a₀ and B₀ move?

Next: [03 — Uniaxial deformation](03-uniaxial-deformation.ipynb), where the
crystal finally gets to move — and to break.